In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
%cd /content

# 切到你的專案資料夾
%cd /content/drive/MyDrive/LSTM_PROGRAM

##################################
# GARCH-X，照計畫書的公式，但效果很差， VAR的 STD 是用 GARCH的SHAPE

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content
/content/drive/MyDrive/LSTM_PROGRAM


In [3]:
!mkdir -p ~/.ssh
!cp /content/drive/MyDrive/.ssh/id_ed25519* ~/.ssh/
!chmod 700 ~/.ssh
!chmod 600 ~/.ssh/id_ed25519

!eval "$(ssh-agent -s)" && ssh-add ~/.ssh/id_ed25519
!ssh-keyscan github.com >> ~/.ssh/known_hosts
!chmod 644 ~/.ssh/known_hosts

!ssh -T git@github.com

!git config --global user.email "joemi7878@gmail.com"
!git config --global user.name "joemi78"

Agent pid 2955
Identity added: /root/.ssh/id_ed25519 (joemi7878@gmail.com)
# github.com:22 SSH-2.0-a73f77f
# github.com:22 SSH-2.0-a73f77f
# github.com:22 SSH-2.0-a73f77f
# github.com:22 SSH-2.0-a73f77f
# github.com:22 SSH-2.0-a73f77f
Hi joemi78! You've successfully authenticated, but GitHub does not provide shell access.


In [4]:
!pip install arch

!sudo apt-get update
!sudo apt-get install -y build-essential python3-dev r-base-dev

!pip install -U jedi
!pip install -U pip setuptools wheel

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
build-essential is already the newest version (12.9ubuntu3).
python3-dev is already the newest

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
import os

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras import mixed_precision

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import precision_score, recall_score, f1_score

from functools import cached_property
from typing import Union, Optional, Dict

from scipy.stats import norm, t as tdist
from scipy.stats import chi2


try:
    from arch import arch_model
    HAS_ARCH = True
except Exception:
    HAS_ARCH = False


#### SET GPU
gpus = tf.config.list_physical_devices('GPU')
print("Num GPUs:", len(gpus), gpus)

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("✓ memory growth set")
    except RuntimeError as e:
        print("⚠️ GPU 已初始化，無法再設定 memory growth：", e)


# （建議）再開啟 XLA 與混合精度
tf.config.optimizer.set_jit(True)
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')

print("是否可用GPU:", tf.test.is_gpu_available())
print("使用中的裝置:", tf.config.list_physical_devices('GPU'))

print("✓ GPU 初始化流程完成")


# 1. 定義檔案路徑
file_paths = {
    # "bonds_day": "./filtered_output/bonds_day_clean_period.csv",
    # "bonds_hour": "./filtered_output/bonds_hour_clean_period.csv",
    # "crypto_day": "./filtered_output/crypto_day_clean_period.csv",
    # "crypto_hour": "./filtered_output/crypto_hour_clean_period.csv",
    # "others_day": "./filtered_output/others_day_clean_period.csv",
    # "others_hour": "./filtered_output/others_hour_clean_period.csv",
    "stock_day":  "./filtered_output/stock_day_fluctuation_aligned.csv"
    # "stock_hour": "./filtered_output/stock_hour_clean_period.csv"
}


def find_date_col(df):
    for col in df.columns:
        if 'date' in col.lower():
            return col
    return df.columns[0]

def read_and_clean(file):
    # ① 正确地读 CSV，不要写 (index=True)
    df = pd.read_csv(file)

    # ② 找到原始时间列名
    date_col = find_date_col(df)

    # ③ 依次尝试各种格式去解析
    parsed = False
    for fmt in [
        '%Y-%m-%d %H:%M:%S',
        '%Y/%m/%d %H:%M:%S',
        '%Y-%m-%d %H:%M',
        '%Y/%m/%d %H:%M',
    ]:
        try:
            df[date_col] = pd.to_datetime(
                df[date_col],
                format=fmt,     # 严格匹配
                errors='raise'  # 抛错就切换下一个 fmt
            )
            parsed = True
            break
        except Exception:
            continue

    # ④ 如果上面都没能解析，再宽松一把
    if not parsed:
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce')

    # ⑤ 把分钟/秒都砍掉，保留到「小时」粒度
    df[date_col] = df[date_col].dt.floor('h').dt.tz_localize(None)

    # ⑥ 把这一列重命名为 DATE
    df = df.rename(columns={date_col: 'DATE'})

    # （可选）如果你想让 DATE 作为索引：
    # df = df.set_index('DATE')

    return df



raw_dfs = {}
for name, path in file_paths.items():
    raw_dfs[name] = read_and_clean(path)
    # print(raw_dfs[name].columns)

# ===== 資料結構：取代 all_data =====
class AssetGroupLite:
    def __init__(self, name, df):
        self.name = name
        df = df.copy()
        df['DATE'] = pd.to_datetime(df['DATE'])
        self.raw = df.set_index('DATE').sort_index()

    @cached_property
    def _close_cols(self):
        return [c for c in self.raw.columns if c.endswith('_CLOSE')]

    @cached_property
    def _vol_cols(self):
        return [c for c in self.raw.columns if c.endswith('_VOLUME')]

    @cached_property
    def close_ln(self):
        if not self._close_cols:
            return self.raw.iloc[[]]
        return np.log(self.raw[self._close_cols]).rename(
            columns=lambda x: x.replace('_CLOSE', '_CLOSE_ln')
        )

    @cached_property
    def close_ln_ret(self):
        if not self._close_cols:
            return self.raw.iloc[[]]
        logp = np.log(self.raw[self._close_cols])
        return (
            logp.diff()
               .rename(columns=lambda x: x.replace('_CLOSE', '_CLOSE_ln_ret'))
               .dropna(how='all')
        )

    @cached_property
    def close_arith_ret(self):
        if not self._close_cols:
            return self.raw.iloc[[]]
        return (
            self.raw[self._close_cols].pct_change()
                .rename(columns=lambda x: x.replace('_CLOSE', '_CLOSE_arith_ret'))
                .dropna(how='all')
        )

class DataRepository:
    REQUIRED_INDEX = "DATE"

    def __init__(self, raw_dfs: dict, check_schema: bool = True):
        self.groups = {}
        for name, df in raw_dfs.items():
            if check_schema:
                assert 'DATE' in df.columns, f"{name}: 缺少 DATE 欄"
            self.groups[name] = AssetGroupLite(name, df)

    # 補上 group()，方便外部與內部呼叫
    def group(self, name: str) -> AssetGroupLite:
        if name not in self.groups:
            raise KeyError(f"Group '{name}' 不存在。可用群組：{list(self.groups.keys())}")
        return self.groups[name]

    def series(self, group: str, series_name: str) -> pd.Series:
        g = self.group(group)
        # 加上 raw → 能抓 OHLCV、IS_TRADING 等原始欄
        search_order = ['close_ln_ret', 'close_arith_ret', 'close_ln', 'raw']

        # 1) 直接命中
        for key in search_order:
            tbl = getattr(g, key)
            if series_name in tbl.columns:
                return tbl[series_name]

        # 2) 容錯：只給 base symbol，自動補候選
        base = (series_name
                .replace('_CLOSE', '')
                .replace('_OPEN', '')
                .replace('_HIGH', '')
                .replace('_LOW', '')
                .replace('_VOLUME', '')
                .replace('_IS_TRADING', '')
                .replace('_CLOSE_ln_ret', '')
                .replace('_CLOSE_arith_ret', '')
                .replace('_CLOSE_ln', ''))
        candidates = [
            f'{base}_CLOSE_ln_ret',
            f'{base}_CLOSE_arith_ret',
            f'{base}_CLOSE_ln',
            f'{base}_OPEN',
            f'{base}_HIGH',
            f'{base}_LOW',
            f'{base}_CLOSE',
            f'{base}_VOLUME',
            f'{base}_IS_TRADING'
        ]
        for cand in candidates:
            for key in search_order:
                tbl = getattr(g, key)
                if cand in tbl.columns:
                    return tbl[cand]

        raise KeyError(f"{group}: 找不到 {series_name} 或候選 {candidates}")

    # 取整張表
    def table(self, group: str, table_name: str) -> pd.DataFrame:
        g = self.group(group)
        if not hasattr(g, table_name):
            raise KeyError(f"{group}: 無表 '{table_name}'。可用表：['close_ln_ret','close_arith_ret','close_ln']")
        return getattr(g, table_name)

    # 若要直接拿 raw 的原始價/量欄位（例如 *_CLOSE 或 *_VOLUME）
    def raw_series(self, group: str, raw_col: str) -> pd.Series:
        g = self.group(group)
        if raw_col not in g.raw.columns:
            raise KeyError(f"{group}: raw 中沒有欄位 {raw_col}")
        return g.raw[raw_col]

repo = DataRepository(raw_dfs)
print("repo success")

Instructions for updating:
Use `tf.config.list_physical_devices('GPU')` instead.


Num GPUs: 0 []
是否可用GPU: False
使用中的裝置: []
✓ GPU 初始化流程完成
repo success


In [6]:
# 1. 定義檔案路徑
garch_file_paths = {
    "garch_params_1": "./garch_data/garch_params_by_day_1.csv"
}

def garch_find_date_col(df):
    for col in df.columns:
        if 'date' in col.lower():
            return col
    return df.columns[0]

def garch_read_and_clean(file):
    # ① 正确地读 CSV，不要写 (index=True)
    df = pd.read_csv(file)

    # ② 找到原始时间列名
    date_col = garch_find_date_col(df)

    # ③ 依次尝试各种格式去解析
    parsed = False
    for fmt in [
        '%Y-%m-%d %H:%M:%S',
        '%Y/%m/%d %H:%M:%S',
        '%Y-%m-%d %H:%M',
        '%Y/%m/%d %H:%M',
    ]:
        try:
            df[date_col] = pd.to_datetime(
                df[date_col],
                format=fmt,     # 严格匹配
                errors='raise'  # 抛错就切换下一个 fmt
            )
            parsed = True
            break
        except Exception:
            continue

    # ④ 如果上面都没能解析，再宽松一把
    if not parsed:
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce')

    # ⑤ 把分钟/秒都砍掉，保留到「小时」粒度
    df[date_col] = df[date_col].dt.floor('h').dt.tz_localize(None)

    # ⑥ 把这一列重命名为 DATE
    df = df.rename(columns={date_col: 'DATE'})

    # （可选）如果你想让 DATE 作为索引：
    # df = df.set_index('DATE')

    return df

garch_dfs = {}
for name, path in garch_file_paths.items():
    garch_dfs[name] = garch_read_and_clean(path)
    print(garch_dfs[name].columns)

Index(['DATE', 'sigma_next', 'omega', 'alpha', 'beta', 'shape', '_fallback',
       'vxreg1_p', 'vxreg2_p', '_conv', '_llh', 'ab'],
      dtype='object')


In [9]:
# ------------------ Features ------------------
def build_feature_df(repo: DataRepository, group: str, symbol: str) -> pd.DataFrame:
    s_open  = repo.raw_series(group, f'{symbol}_OPEN').asfreq('D')
    s_high  = repo.raw_series(group, f'{symbol}_HIGH').asfreq('D')
    s_low   = repo.raw_series(group, f'{symbol}_LOW').asfreq('D')
    s_close = repo.raw_series(group, f'{symbol}_CLOSE').asfreq('D')
    s_vol   = repo.raw_series(group, f'{symbol}_VOLUME').asfreq('D')
    s_flag = repo.raw_series(group, f'{symbol}_IS_TRADING').asfreq('D')
    s_lnrt  = repo.series(group, f'{symbol}_CLOSE_ln_ret').asfreq('D')
    s_ln  = repo.series(group, f'{symbol}_CLOSE_ln').asfreq('D')
    df = pd.concat([
        s_lnrt.rename(f'{symbol}_LN_RET'),
        s_open.rename(f'{symbol}_OPEN'),
        s_high.rename(f'{symbol}_HIGH'),
        s_low.rename(f'{symbol}_LOW'),
        s_close.rename(f'{symbol}_CLOSE'),
        s_vol.rename(f'{symbol}_VOLUME'),
        s_flag.rename(f'{symbol}_IS_TRADING'),
        s_ln.rename(f'{symbol}_CLOSE_LN')
        ], axis=1)
    df = df.apply(pd.to_numeric, errors='coerce')
    return df.dropna(how='any')



# ------------------ Config ------------------
ASSET_SYMBOL_ES1 = 'ES1'
ASSET_SYMBOL_VIX = 'VIX'
GROUP_DAY   = 'stock_day'
TARGET_START_STR = '2005-08-01'
TARGET_END_STR   = '2025-06-30'

# # 將輸出在特定的區間
# EXPORT_START = pd.to_datetime('2005-06-01')
# EXPORT_END   = pd.to_datetime('2025-06-30')

# GARCH_WINDOW_DAY = 252
VOL_WINDOW_DAY = 20


feat_ES1 = build_feature_df(repo, GROUP_DAY, ASSET_SYMBOL_ES1)
feat_VIX = build_feature_df(repo, GROUP_DAY, ASSET_SYMBOL_VIX)

df_day = feat_ES1.join(feat_VIX, how='left')

# --- 讓索引成為 datetime（很重要） ---
df_day.index = pd.to_datetime(df_day.index, errors='coerce')
df_day = df_day.sort_index()
df_day = df_day.loc[df_day['ES1_IS_TRADING'] == 1]

# print(df_day.head())
# print(df_day.columns)

df_day.to_csv("./filtered_output/df_day.csv", index=True)


PRED_START = pd.to_datetime(TARGET_START_STR)
PRED_END   = pd.to_datetime(TARGET_END_STR)

# 把回測期限制在資料範圍內
data_start = df_day.index.min()
data_end   = df_day.index.max()
if PRED_START < data_start: PRED_START = data_start
if PRED_END   > data_end:   PRED_END   = data_end

# 若 PRED_START 不是可用交易日，推到 >= PRED_START 的第一個交易日
try:
    PRED_START = df_day.index[df_day.index.searchsorted(PRED_START)]
except Exception:
    # 若整段都沒資料，直接報錯
    raise RuntimeError("資料期間與回測期間沒有交集，請調整 TARGET_START/END。")


first_needed = PRED_START - pd.Timedelta(days= VOL_WINDOW_DAY)
es1_min = df_day.index.min()
ES1_close_series = df_day['ES1_CLOSE']
if es1_min > first_needed:
    raise RuntimeError(f"Insufficient history: need <= {first_needed}, have from {ES1_close_series.index.min()}")



# ------------------ Model ------------------


# 1) 準備母表（確保排序與期間）
# 全歷史的交易日索引（只留 ES1 開市）
dates_all = df_day.index
# print(dates_all)

# 回測區間（仍然要取，決定哪些 t 需要預測）
mask_period = (df_day.index >= PRED_START) & (df_day.index <= PRED_END)
dates_period = df_day.index[mask_period]

# 2) 方便取用的短名
feat_df = df_day  # 與舊程式一致
rows = []




# 將 GARCH 數據加進 feat_df
garch_df = next(iter(garch_dfs.values())).copy()

# --- 1) 整理 garch_df：DATE -> index，只留兩欄，並轉成「日」粒度 ---
garch_small = (
    garch_df[["DATE", "sigma_next", "shape"]]
      .assign(DATE=lambda d: pd.to_datetime(d["DATE"], errors="coerce").dt.floor("D"))  # ✅ 改這行
      .dropna(subset=["DATE"])
      .set_index("DATE")
      .sort_index()
)

# 若同一天有重複（有時候資料會），保留最後一筆
garch_small = garch_small[~garch_small.index.duplicated(keep="last")]

# --- 2) feat_df 的 index 也轉成「日」粒度（避免含時分秒對不到）---
feat_df = feat_df.copy()
feat_df.index = pd.to_datetime(feat_df.index, errors="coerce").floor("D")  # ✅ 改這行
feat_df = feat_df.sort_index()

# --- 3) 以 feat_df 為主表做 join（DATE 對齊）---
feat_df = feat_df.join(garch_small, how="left")




# ================================
# Volatility: SMA / EWMA(A/B) / GK
# ================================

EWMA_LAMBDA = 0.94

# ----------------------------
# 0) 準備資料
# ----------------------------
vol_df = feat_df.copy()
vol_df = vol_df.sort_index()


# 關鍵欄位
ret_col   = 'ES1_LN_RET'
open_col  = 'ES1_OPEN'
high_col  = 'ES1_HIGH'
low_col   = 'ES1_LOW'
close_col = 'ES1_CLOSE'

required_cols = [ret_col, open_col, high_col, low_col, close_col]
missing_cols = [c for c in required_cols if c not in vol_df.columns]
if missing_cols:
    raise KeyError(f"缺少必要欄位: {missing_cols}")

# 轉成 numeric
for c in required_cols:
    vol_df[c] = pd.to_numeric(vol_df[c], errors='coerce')

# log return
vol_df['ret'] = vol_df[ret_col]

# ----------------------------
# 1) SMA (傳統版：同一窗口平均數)
# ----------------------------
rolling_mean_20 = vol_df['ret'].rolling(
    window=VOL_WINDOW_DAY,
    min_periods=VOL_WINDOW_DAY
).mean()

rolling_mean_sq_20 = (vol_df['ret'] ** 2).rolling(
    window=VOL_WINDOW_DAY,
    min_periods=VOL_WINDOW_DAY
).mean()

# 傳統 rolling variance: E[r^2] - (E[r])^2
vol_df['sma_var_20'] = rolling_mean_sq_20 - (rolling_mean_20 ** 2)

# 避免浮點誤差造成極小負值
vol_df['sma_var_20'] = vol_df['sma_var_20'].clip(lower=0)

vol_df['sma_vol_20'] = np.sqrt(vol_df['sma_var_20'])

# ----------------------------
# 1) SMA predict (傳統版：同一窗口平均數)
# ----------------------------

ret_lag1 = vol_df['ret'].shift(1)

vol_df['sma_var_20_predict'] = ret_lag1.rolling(
    window=VOL_WINDOW_DAY,
    min_periods=VOL_WINDOW_DAY
).var(ddof=0)

vol_df['sma_vol_20_predict'] = np.sqrt(vol_df['sma_var_20_predict'])

# ----------------------------
# 2) EWMA A方案：保留平均數
#    用 rolling mean(20) 當期中心
# ----------------------------
ret_centered_A = vol_df['ret'] - rolling_mean_20

ewma_var_A = np.full(len(vol_df), np.nan)
ret_centered_A_values = ret_centered_A.to_numpy()

# 初始值：第一個可用20日視窗的 sample variance
first_valid_idx_A = None
for i in range(VOL_WINDOW_DAY - 1, len(vol_df)):
    window_vals = ret_centered_A_values[i - VOL_WINDOW_DAY + 1 : i + 1]
    if np.isfinite(window_vals).all():
        first_valid_idx_A = i
        ewma_var_A[i] = np.mean(window_vals ** 2)
        break

# 遞迴
if first_valid_idx_A is not None:
    for i in range(first_valid_idx_A + 1, len(vol_df)):
        x_prev = ret_centered_A_values[i - 1]
        prev_var = ewma_var_A[i - 1]
        if np.isfinite(x_prev) and np.isfinite(prev_var):
            ewma_var_A[i] = EWMA_LAMBDA * prev_var + (1 - EWMA_LAMBDA) * (x_prev ** 2)

vol_df['ewmaA_var_20'] = ewma_var_A
vol_df['ewmaA_vol_20'] = np.sqrt(vol_df['ewmaA_var_20'])


# ----------------------------
# 3) EWMA B方案：不保留平均數
#    直接用 r_{t-1}^2
# ----------------------------
ewma_var_B = np.full(len(vol_df), np.nan)
ret_values = vol_df['ret'].to_numpy()

# 初始值：第一個20日視窗的平方報酬平均
first_valid_idx_B = None
for i in range(VOL_WINDOW_DAY - 1, len(vol_df)):
    window_vals = ret_values[i - VOL_WINDOW_DAY + 1 : i + 1]
    if np.isfinite(window_vals).all():
        first_valid_idx_B = i
        ewma_var_B[i] = np.mean(window_vals ** 2)
        break

# 遞迴
if first_valid_idx_B is not None:
    for i in range(first_valid_idx_B + 1, len(vol_df)):
        x_prev = ret_values[i - 1]
        prev_var = ewma_var_B[i - 1]
        if np.isfinite(x_prev) and np.isfinite(prev_var):
            ewma_var_B[i] = EWMA_LAMBDA * prev_var + (1 - EWMA_LAMBDA) * (x_prev ** 2)

vol_df['ewmaB_var_20'] = ewma_var_B
vol_df['ewmaB_vol_20'] = np.sqrt(vol_df['ewmaB_var_20'])


# ----------------------------
# 4) GK (Garman-Klass)
#    先算單日 GK variance，再視窗20日平均
# ----------------------------
log_hl = np.log(vol_df[high_col] / vol_df[low_col])
log_co = np.log(vol_df[close_col] / vol_df[open_col])

vol_df['gk_var_daily'] = 0.5 * (log_hl ** 2) - (2 * np.log(2) - 1) * (log_co ** 2)

# 理論上很少，但若數值誤差出現負值，截到0
vol_df['gk_var_daily'] = vol_df['gk_var_daily'].clip(lower=0)

# 20日 GK variance / volatility
vol_df['gk_var_20'] = vol_df['gk_var_daily'].rolling(window=VOL_WINDOW_DAY, min_periods=VOL_WINDOW_DAY).mean()
vol_df['gk_vol_20'] = np.sqrt(vol_df['gk_var_20'])


# ----------------------------
# 5) GARCH 整理
#    你現有 garch_small 已 join 到 feat_df
#    這裡沿用 sigma_next 當 GARCH volatility
# ----------------------------
if 'sigma_next' in vol_df.columns:
    vol_df['garch_vol'] = pd.to_numeric(vol_df['sigma_next'], errors='coerce')
    vol_df['garch_var'] = vol_df['garch_vol'] ** 2
else:
    vol_df['garch_vol'] = np.nan
    vol_df['garch_var'] = np.nan


# ----------------------------
# 6) 取目標區間 TARGET_START_STR ~ TARGET_END_STR
# ----------------------------
target_start = pd.to_datetime(TARGET_START_STR)
target_end   = pd.to_datetime(TARGET_END_STR)

vol_result_df = vol_df.loc[(vol_df.index >= target_start) & (vol_df.index <= target_end)].copy()

# 若 index 含重複，保留最後一筆
vol_result_df = vol_result_df[~vol_result_df.index.duplicated(keep='last')]

# ----------------------------
# 7) 輸出最終 DataFrame
# ----------------------------
vol_result_df = vol_result_df[[
    ret_col,
    'sma_var_20',   'sma_vol_20',
    'sma_var_20_predict',   'sma_vol_20_predict',
    'ewmaA_var_20', 'ewmaA_vol_20',
    'ewmaB_var_20', 'ewmaB_vol_20',
    'gk_var_daily',
    'gk_var_20',    'gk_vol_20',
    'garch_var',    'garch_vol',
    'shape' if 'shape' in vol_result_df.columns else ret_col
]].copy()

# 若最後一欄因為 shape 不存在而重複加到 ret_col，把重複欄去掉
vol_result_df = vol_result_df.loc[:, ~vol_result_df.columns.duplicated()].copy()

# 將 index 轉回 DATE 欄，方便後續 merge / export / 統計
vol_result_df = vol_result_df.reset_index().rename(columns={'index': 'DATE'})

print(vol_result_df.head())
print(vol_result_df.tail())
print(vol_result_df.columns.tolist())
print(vol_result_df.shape)

# 如要存檔
vol_result_df.to_csv("./garch_data/es1_volatility_all_methods.csv", index=False)




        DATE  ES1_LN_RET  sma_var_20  sma_vol_20  sma_var_20_predict  \
0 2005-08-01    0.000606    0.000026    0.005056            0.000026   
1 2005-08-02    0.006042    0.000025    0.004981            0.000026   
2 2005-08-03    0.002206    0.000019    0.004399            0.000025   
3 2005-08-04   -0.007845    0.000024    0.004875            0.000019   
4 2005-08-05   -0.006686    0.000021    0.004619            0.000024   

   sma_vol_20_predict  ewmaA_var_20  ewmaA_vol_20  ewmaB_var_20  ewmaB_vol_20  \
0            0.005074      0.000028      0.005257      0.000029      0.005396   
1            0.005056      0.000026      0.005102      0.000027      0.005234   
2            0.004981      0.000026      0.005072      0.000028      0.005286   
3            0.004399      0.000024      0.004918      0.000027      0.005153   
4            0.004875      0.000028      0.005283      0.000029      0.005353   

   gk_var_daily  gk_var_20  gk_vol_20  garch_var  garch_vol      shape  
0      

In [10]:
import numpy as np
import pandas as pd

from scipy.stats import jarque_bera
from statsmodels.stats.diagnostic import acorr_ljungbox

import matplotlib.pyplot as plt

# =========================
# 0) Setup
# =========================
var_cols = [
    'sma_var_20',
    'sma_var_20_predict',
    'ewmaA_var_20',
    'ewmaB_var_20',
    'gk_var_daily',
    'gk_var_20',
    'garch_var'
]
var_cols = [c for c in var_cols if c in vol_result_df.columns]

var_df = vol_result_df[['DATE'] + var_cols].copy()
var_df['DATE'] = pd.to_datetime(var_df['DATE'], errors='coerce')

# =========================
# 1) Descriptive statistics
# =========================
desc_stats = var_df[var_cols].agg(['count', 'mean', 'std', 'min', 'max']).T
quantiles = var_df[var_cols].quantile([0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]).T
quantiles.columns = ['q01', 'q05', 'q25', 'q50', 'q75', 'q95', 'q99']
desc_stats['skew'] = var_df[var_cols].skew()
desc_stats['kurt'] = var_df[var_cols].kurt()

summary_stats_df = pd.concat([desc_stats, quantiles], axis=1)
summary_stats_df = summary_stats_df[
    ['count', 'mean', 'std', 'min', 'q01', 'q05', 'q25', 'q50', 'q75', 'q95', 'q99', 'max', 'skew', 'kurt']
]

# =========================
# 2) Jarque-Bera
# =========================
jb_rows = []
for col in var_cols:
    s = var_df[col].dropna()
    if len(s) > 0:
        jb_stat, jb_p = jarque_bera(s)
        jb_rows.append({
            'variable': col,
            'n': len(s),
            'jb_stat': jb_stat,
            'jb_pvalue': jb_p,
            'normal_at_5pct': 'Yes' if jb_p >= 0.05 else 'No'
        })
jb_df = pd.DataFrame(jb_rows)

# =========================
# 3) Correlations
# =========================
pearson_corr_df = var_df[var_cols].corr(method='pearson')
spearman_corr_df = var_df[var_cols].corr(method='spearman')

# =========================
# 4) Ljung-Box
# =========================
lb_rows = []
for col in var_cols:
    s = var_df[col].dropna()
    if len(s) > 30:
        lb_5 = acorr_ljungbox(s, lags=[5], return_df=True)
        lb_10 = acorr_ljungbox(s, lags=[10], return_df=True)
        lb_20 = acorr_ljungbox(s, lags=[20], return_df=True)

        lb_rows.append({
            'variable': col,
            'acf_lag1': s.autocorr(lag=1),
            'acf_lag5': s.autocorr(lag=5),
            'acf_lag10': s.autocorr(lag=10),
            'lb_stat_5': lb_5['lb_stat'].iloc[0],
            'lb_pvalue_5': lb_5['lb_pvalue'].iloc[0],
            'lb_stat_10': lb_10['lb_stat'].iloc[0],
            'lb_pvalue_10': lb_10['lb_pvalue'].iloc[0],
            'lb_stat_20': lb_20['lb_stat'].iloc[0],
            'lb_pvalue_20': lb_20['lb_pvalue'].iloc[0],
        })
lb_df = pd.DataFrame(lb_rows)

# =========================
# 5) Pairwise difference summary
# =========================
pair_rows = []
for i in range(len(var_cols)):
    for j in range(i + 1, len(var_cols)):
        c1 = var_cols[i]
        c2 = var_cols[j]

        tmp = var_df[[c1, c2]].dropna().copy()
        if len(tmp) == 0:
            continue

        diff = tmp[c1] - tmp[c2]

        pair_rows.append({
            'var1': c1,
            'var2': c2,
            'n': len(diff),
            'mean_diff': diff.mean(),
            'std_diff': diff.std(),
            'mae': diff.abs().mean(),
            'rmse': np.sqrt(np.mean(diff**2)),
            'corr_pearson': tmp[c1].corr(tmp[c2], method='pearson'),
            'corr_spearman': tmp[c1].corr(tmp[c2], method='spearman')
        })
pairwise_diff_df = pd.DataFrame(pair_rows)

# =========================
# 6) Output
# =========================
print("\n[Summary stats]")
print(summary_stats_df.round(8))

print("\n[Jarque-Bera]")
print(jb_df.round(6))

print("\n[Pearson correlation]")
print(pearson_corr_df.round(4))

print("\n[Spearman correlation]")
print(spearman_corr_df.round(4))

print("\n[Ljung-Box]")
print(lb_df.round(6))

print("\n[Pairwise difference summary]")
print(pairwise_diff_df.round(8))

# =========================
# 7) Save
# =========================
summary_stats_df.to_csv('./garch_data/summary_stats_var.csv', encoding='utf-8-sig')
jb_df.to_csv('./garch_data/jarque_bera_var.csv', index=False, encoding='utf-8-sig')
pearson_corr_df.to_csv('./garch_data/pearson_corr_var.csv', encoding='utf-8-sig')
spearman_corr_df.to_csv('./garch_data/spearman_corr_var.csv', encoding='utf-8-sig')
lb_df.to_csv('./garch_data/ljung_box_var.csv', index=False, encoding='utf-8-sig')
pairwise_diff_df.to_csv('./garch_data/pairwise_diff_var.csv', index=False, encoding='utf-8-sig')

print("\nAll variance statistics tables saved.")


# 確保 DATE 是 datetime
var_df['DATE'] = pd.to_datetime(var_df['DATE'], errors='coerce')
var_df = var_df.sort_values('DATE').dropna(subset=['DATE']).copy()

# 輸出資料夾
out_dir = './garch_data'
os.makedirs(out_dir, exist_ok=True)

# 你指定的欄位
group1 = ['sma_var_20', 'gk_var_daily']
group2 = ['sma_var_20_predict', 'ewmaA_var_20', 'ewmaB_var_20', 'garch_var']

# 檢查欄位
need_cols = ['DATE'] + group1 + group2
missing_cols = [c for c in need_cols if c not in var_df.columns]
if missing_cols:
    raise KeyError(f"缺少欄位: {missing_cols}")

# -------------------------
# 1) 各別單張圖存檔
# -------------------------
for col in group1 + group2:
    plot_df = var_df[['DATE', col]].dropna().copy()

    plt.figure(figsize=(12, 5))
    plt.plot(plot_df['DATE'], plot_df[col], linewidth=1)
    plt.title(col)
    plt.xlabel('Date')
    plt.ylabel(col)
    plt.tight_layout()
    plt.savefig(f'{out_dir}/{col}.png', dpi=200, bbox_inches='tight')
    plt.close()

# -------------------------
# 2) 合併比較圖存檔
# -------------------------
plt.figure(figsize=(12, 5))
for col in group1:
    plot_df = var_df[['DATE', col]].dropna().copy()
    plt.plot(plot_df['DATE'], plot_df[col], linewidth=1, label=col)

plt.title('sma_var_20_vs_gk_var_daily')
plt.xlabel('Date')
plt.ylabel('Variance')
plt.legend()
plt.tight_layout()
plt.savefig(f'{out_dir}/sma_var_20_vs_gk_var_daily.png', dpi=200, bbox_inches='tight')
plt.close()

plt.figure(figsize=(12, 5))
for col in group2:
    plot_df = var_df[['DATE', col]].dropna().copy()
    plt.plot(plot_df['DATE'], plot_df[col], linewidth=1, label=col)

plt.title('sma_var_20_predict_vs_ewmaA_vs_ewmaB_vs_garch_var')
plt.xlabel('Date')
plt.ylabel('Variance')
plt.legend()
plt.tight_layout()
plt.savefig(f'{out_dir}/sma_var_20_predict_vs_ewmaA_vs_ewmaB_vs_garch_var.png', dpi=200, bbox_inches='tight')
plt.close()

print(f'圖已存到: {out_dir}')


[Summary stats]
                     count      mean       std       min       q01       q05  \
sma_var_20          5030.0  0.000145  0.000321  0.000003  0.000008  0.000014   
sma_var_20_predict  5030.0  0.000145  0.000321  0.000003  0.000008  0.000014   
ewmaA_var_20        5030.0  0.000143  0.000275  0.000007  0.000011  0.000017   
ewmaB_var_20        5030.0  0.000150  0.000291  0.000007  0.000013  0.000019   
gk_var_daily        5030.0  0.000137  0.000345  0.000001  0.000005  0.000010   
gk_var_20           5030.0  0.000137  0.000252  0.000007  0.000015  0.000022   
garch_var           5030.0  0.000158  0.000359  0.000011  0.000015  0.000024   

                         q25       q50       q75       q95       q99  \
sma_var_20          0.000032  0.000067  0.000137  0.000395  0.002184   
sma_var_20_predict  0.000032  0.000067  0.000137  0.000395  0.002184   
ewmaA_var_20        0.000040  0.000068  0.000138  0.000448  0.001950   
ewmaB_var_20        0.000042  0.000071  0.000144  0.00